In [1]:
import pandas as pd
data = pd.read_csv('./stroke-dataset.csv')
data

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [2]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

data.fillna(0, inplace=True)

X = data.drop(['stroke'], axis=1)
y = data['stroke']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

X.smoking_status.value_counts()

smoking_status
never smoked       1892
Unknown            1544
formerly smoked     885
smokes              789
Name: count, dtype: int64

In [3]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline

nominal_columns = ['work_type', 'gender']
ordinal_or_binary_columns = ['Residence_type', 'ever_married']
irrelevant_columns = ['id']

ordinal_encoder = OrdinalEncoder(
    categories=[
        ['Rural', 'Urban'],
        ['No', 'Yes']
    ]
)

feature_encoder = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'), nominal_columns),
    (ordinal_encoder, ordinal_or_binary_columns),
    ('drop', irrelevant_columns),
    remainder='passthrough'
)

scaler = MinMaxScaler()

## Assigns all instances of 'Unknown' to the mean value of all known smoking statuses assuming:
# never smoked = 0
# formerly smoked = 1
# smokes = 2
class SmokingStatusEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='smoking_status'):
        self.column = column
        self.smoking_map = {
            'never smoked': 0,
            'formerly smoked': 1,
            'smokes': 2
        }
        self.mean_value = None

    def fit(self, X, y=None):
        # Calculate mean only on known smoking statuses
        known = X[X[self.column].isin(self.smoking_map.keys())]
        encoded = known[self.column].map(self.smoking_map)
        self.mean_value = encoded.mean()
        return self

    def transform(self, X, y=None):
        X = X.copy()
        # Replace with mean if 'Unknown', otherwise map value
        X[self.column] = X[self.column].apply(
            lambda val: self.smoking_map.get(val, self.mean_value)
        )
        return X

class PreprocessingTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_encoded = feature_encoder.fit_transform(X)
        scaler.fit(X_encoded)
        return self

    def transform(self, X, y=None):
        X_scaled = scaler.transform(feature_encoder.transform(X))
        # Return the 
        return pd.DataFrame(X_scaled, index=X.index)

preprocessing_pipeline = make_pipeline(SmokingStatusEncoder(), PreprocessingTransformer())
preprocessing_pipeline.fit_transform(X_train)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
2332,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.841309,0.0,1.0,0.661112,0.437500,1.000000
4347,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.365234,0.0,0.0,0.154372,0.464139,0.000000
3156,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.353027,0.0,0.0,0.064629,0.274590,0.000000
1564,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.304199,0.0,0.0,0.063060,0.564549,0.500000
12,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.658203,0.0,0.0,0.228003,0.279713,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.682617,0.0,0.0,0.120303,0.275615,0.000000
3276,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.963379,0.0,0.0,0.107100,0.327869,0.345063
1318,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.609375,0.0,1.0,0.141723,0.326844,0.000000
723,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.609375,0.0,0.0,0.104007,0.250000,1.000000


In [4]:
from sklearn.metrics import get_scorer
from sklearn.model_selection import StratifiedKFold
import numpy as np

from sklearn.metrics import fbeta_score

def f2_metric(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)

def cross_validate_and_predict(model, X_train, y_train, metrics, cv_folds=10, random_state=1234):
    # Convert metrics to functions if they are strings
    metric_names = [metric.__name__ if callable(metric) else metric for metric in metrics]
    metric_fns = [metric if callable(metric) else get_scorer(metric)._score_func for metric in metrics]
    kfold = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    y_preds = np.zeros(y_train.shape)
    results = []
    fold = 1
    for train_idx, test_idx in kfold.split(X_train, y_train):
        model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        y_pred = model.predict(X_train.iloc[test_idx])
        y_preds[test_idx] = y_pred
        fold_results = {'fold': fold}
        for metric_name, metric_fn in zip(metric_names, metric_fns):
            value = metric_fn(y_train.iloc[test_idx], y_pred)
            fold_results[metric_name] = value
        results.append(fold_results)
        fold += 1
    return pd.DataFrame(results), y_preds

In [5]:
# Add imports!!
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

scoring_metrics = []

scoring_metrics.append('accuracy')
scoring_metrics.append('precision')
scoring_metrics.append('recall')
scoring_metrics.append('f1')
scoring_metrics.append(f2_metric)
scoring_metrics.append('average_precision')

knn_classifier = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='minkowski', p=2, algorithm='auto')
knn_pipeline = make_pipeline(SmokingStatusEncoder(), PreprocessingTransformer(), knn_classifier)

knn_performance, knn_pred = cross_validate_and_predict(
    knn_pipeline, X_train, y_train, scoring_metrics
)

print(classification_report(y_train, knn_pred))
print(confusion_matrix(y_train, knn_pred))

knn_performance

c:\Users\johnw\MyDocuments\Schoolwork\2025_Spring\CSC_522_(ALDA)\Final_Proj\alda-project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3890
           1       0.21      0.02      0.03       198

    accuracy                           0.95      4088
   macro avg       0.58      0.51      0.50      4088
weighted avg       0.92      0.95      0.93      4088

[[3879   11]
 [ 195    3]]


c:\Users\johnw\MyDocuments\Schoolwork\2025_Spring\CSC_522_(ALDA)\Final_Proj\alda-project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,fold,accuracy,precision,recall,f1,f2_metric,average_precision
0,1,0.953545,1.000000,0.05,0.095238,0.061728,0.096455
1,2,0.953545,1.000000,0.05,0.095238,0.061728,0.096455
2,3,0.946210,0.000000,0.00,0.000000,0.000000,0.048900
3,4,0.951100,0.000000,0.00,0.000000,0.000000,0.048900
4,5,0.948655,0.333333,0.05,0.086957,0.060241,0.063121
5,6,0.948655,0.000000,0.00,0.000000,0.000000,0.048900
6,7,0.948655,0.000000,0.00,0.000000,0.000000,0.048900
7,8,0.951100,0.000000,0.00,0.000000,0.000000,0.048900
8,9,0.948529,0.000000,0.00,0.000000,0.000000,0.046569
9,10,0.946078,0.000000,0.00,0.000000,0.000000,0.046569


In [6]:
### Oversampling with SMOTE
 
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.metrics import precision_score, recall_score, f1_score

def smote_cross_validate_and_predict(model, preprocessor, X, y, metrics, cv_folds=10, random_state=1234):
    # Convert metrics to functions if they are strings
    metric_names = [metric.__name__ if callable(metric) else metric for metric in metrics]
    metric_fns = [metric if callable(metric) else get_scorer(metric)._score_func for metric in metrics]
    kfold = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    y_preds = pd.Series(index=y.index, dtype=float)
    results = []
    fold = 1
    for train_idx, val_idx in kfold.split(X, y):
        original_val_indices = y.iloc[val_idx].index

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        X_train = X_train.reset_index(drop=True)
        y_train = y_train.reset_index(drop=True)
        X_val = X_val.reset_index(drop=True)
        y_val = y_val.reset_index(drop=True)

        # Fit + transform on training data only
        fold_preprocessor = clone(preprocessor)
        X_train_processed = fold_preprocessor.fit_transform(X_train)
        X_val_processed = fold_preprocessor.transform(X_val)

        # Apply SMOTE on the processed training data
        smote = SMOTE(random_state=random_state)
        X_train_balanced, y_train_balanced = smote.fit_resample(X_train_processed, y_train)

        # Fit the model on balanced training data
        model.fit(X_train_balanced, y_train_balanced)

        y_pred = model.predict(X_val_processed)
        y_preds.loc[original_val_indices] = y_pred
        
        fold_results = {'fold': fold}
        for metric_name, metric_fn in zip(metric_names, metric_fns):
            value = metric_fn(y.iloc[val_idx], y_pred)
            fold_results[metric_name] = value

        fold_results['prec_0'] = precision_score(y_val, y_pred, pos_label=0)
        fold_results['rec_0'] = recall_score(y_val, y_pred, pos_label=0)
        fold_results['f1_0'] = f1_score(y_val, y_pred, pos_label=0)
        fold_results['f2_0'] = fbeta_score(y_val, y_pred, beta=2, pos_label=0)

        results.append(fold_results)
        fold += 1
    return pd.DataFrame(results), y_preds

In [7]:
class FullPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.smoking = SmokingStatusEncoder()
        self.features = PreprocessingTransformer()

    def fit(self, X, y=None):
        X_smoke = self.smoking.fit_transform(X)
        self.features.fit(X_smoke)
        return self

    def transform(self, X, y=None):
        X_smoke = self.smoking.transform(X)
        return self.features.transform(X_smoke)

knn = KNeighborsClassifier(n_neighbors=5)
preprocessor = FullPreprocessor()

knn_smote_performance, knn_smote_pred = smote_cross_validate_and_predict(
    knn, preprocessor, X_train, y_train, scoring_metrics
)

print(classification_report(y_train, knn_smote_pred))
print(confusion_matrix(y_train, knn_smote_pred))

knn_smote_df = pd.DataFrame(knn_smote_performance)
knn_smote_df = knn_smote_df.sort_values(by='f2_metric', ascending=False)
knn_smote_df.head(10)

              precision    recall  f1-score   support

           0       0.96      0.87      0.91      3890
           1       0.13      0.38      0.19       198

    accuracy                           0.84      4088
   macro avg       0.55      0.62      0.55      4088
weighted avg       0.92      0.84      0.88      4088

[[3367  523]
 [ 123   75]]


,fold,accuracy,precision,recall,f1,f2_metric,average_precision,prec_0,rec_0,f1_0,f2_0
3,4,0.880196,0.191489,0.450000,0.268657,0.354331,0.113065,0.969613,0.902314,0.934754,0.915016
5,6,0.853301,0.155172,0.450000,0.230769,0.326087,0.096722,0.968661,0.874036,0.918919,0.891453
8,9,0.838235,0.138462,0.473684,0.214286,0.319149,0.090097,0.970845,0.856041,0.909836,0.876777
0,1,0.855746,0.145455,0.400000,0.213333,0.296296,0.087522,0.966102,0.879177,0.920592,0.895288
6,7,0.848411,0.137931,0.400000,0.205128,0.289855,0.084512,0.965812,0.871465,0.916216,0.888831
4,5,0.841076,0.131148,0.400000,0.197531,0.283688,0.081799,0.965517,0.863753,0.911805,0.882353
1,2,0.843521,0.120690,0.350000,0.179487,0.253623,0.074026,0.962963,0.868895,0.913514,0.886209
7,8,0.833741,0.100000,0.300000,0.150000,0.214286,0.064230,0.959885,0.861183,0.907859,0.879265
9,10,0.803922,0.082192,0.315789,0.130435,0.201342,0.057818,0.961194,0.827763,0.889503,0.851401
2,3,0.821516,0.079365,0.250000,0.120482,0.174825,0.056516,0.956647,0.850900,0.900680,0.870137


In [8]:
### Hyperparameter Tuning

from itertools import product

k_vals = [48, 49, 50, 51, 52, 53, 54]
weight_vals = ['uniform', 'distance']
p_vals = [1, 2, 3]

results = []
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'f2_metric', 'average_precision', 'prec_0', 'rec_0', 'f1_0', 'f2_0']

for k, w, p in product(k_vals, weight_vals, p_vals):
    model = KNeighborsClassifier(n_neighbors=k, weights=w, p=p)

    perf, preds = smote_cross_validate_and_predict(model, preprocessor, X_train, y_train, scoring_metrics)

    mean_metrics = perf[metric_names].mean()

    results.append({
        'k': k,
        'weights': w,
        'p': p,
        **mean_metrics.to_dict()
    })

results_df = pd.DataFrame(results)

In [9]:
results_df = results_df.sort_values(by='f2_metric', ascending=False)
results_df.head(10)

,k,weights,p,accuracy,precision,recall,f1,f2_metric,average_precision,prec_0,rec_0,f1_0,f2_0
30,53,uniform,1,0.752195,0.111199,0.580263,0.186090,0.313113,0.086301,0.972782,0.760925,0.853446,0.795338
22,51,distance,2,0.753170,0.111064,0.575263,0.185792,0.312174,0.085411,0.972435,0.762211,0.854288,0.796501
11,49,distance,3,0.746075,0.109286,0.585263,0.183797,0.311624,0.084730,0.972830,0.754242,0.849383,0.789566
16,50,distance,2,0.751946,0.110681,0.575263,0.185220,0.311457,0.085179,0.972385,0.760925,0.853422,0.795351
32,53,uniform,3,0.711587,0.102914,0.636053,0.176859,0.311340,0.084332,0.974829,0.715424,0.824717,0.755393
31,53,uniform,2,0.722105,0.104410,0.620789,0.178467,0.311111,0.084562,0.974245,0.727249,0.832392,0.765882
28,52,distance,2,0.751457,0.110396,0.575263,0.184791,0.310920,0.084941,0.972388,0.760411,0.853083,0.794894
7,49,uniform,2,0.724306,0.104768,0.615789,0.178747,0.310626,0.084355,0.973980,0.729820,0.833965,0.768123
10,49,distance,2,0.754394,0.110804,0.570263,0.185164,0.310574,0.085017,0.972175,0.763753,0.855154,0.797813
34,53,distance,2,0.751947,0.110071,0.575263,0.184364,0.310488,0.084492,0.972458,0.760925,0.853451,0.795362


In [10]:
from sklearn.dummy import DummyClassifier

#dummy = DummyClassifier(strategy='constant', constant=1)
dummy = DummyClassifier(strategy='uniform')
dummy_pipeline = make_pipeline(SmokingStatusEncoder(), PreprocessingTransformer(), dummy)

dummy_performance, dummy_pred = cross_validate_and_predict(
    dummy_pipeline, X_train, y_train, scoring_metrics
)

dummy_df = pd.DataFrame(dummy_performance)
dummy_df = dummy_df.sort_values(by='f2_metric', ascending=False)
dummy_df.head(10)

,fold,accuracy,precision,recall,f1,f2_metric,average_precision
9,10,0.492647,0.064815,0.736842,0.119149,0.239726,0.060013
1,2,0.484108,0.063927,0.700000,0.117155,0.234114,0.059419
2,3,0.498778,0.061611,0.650000,0.112554,0.223368,0.057162
6,7,0.498778,0.053140,0.550000,0.096916,0.191638,0.051232
4,5,0.511002,0.050000,0.500000,0.090909,0.178571,0.049450
8,9,0.509804,0.045226,0.473684,0.082569,0.163636,0.045933
3,4,0.550122,0.044444,0.400000,0.080000,0.153846,0.047118
5,6,0.503667,0.040201,0.400000,0.073059,0.143369,0.045420
7,8,0.530562,0.037634,0.350000,0.067961,0.131579,0.044957
0,1,0.484108,0.034146,0.350000,0.062222,0.122807,0.043736


In [11]:
### Final Model for this Milestone

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, average_precision_score

final_knn = KNeighborsClassifier(n_neighbors=53, weights='uniform', p=1)
preprocessor = FullPreprocessor()

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

X_train_bal, y_train_bal = SMOTE(random_state=1234).fit_resample(X_train_proc, y_train)

final_knn.fit(X_train_bal, y_train_bal)

# Your predicted labels
y_train_pred = final_knn.predict(X_train_bal)
y_test_pred = final_knn.predict(X_test_proc)

# For average_precision_score, we need predicted probabilities
y_test_prob = final_knn.predict_proba(X_test_proc)[:, 1]  # Prob for class 1

# Build a name-function mapping (handles string and callable metrics)
metrics = {
    'accuracy': accuracy_score,
    'precision': precision_score,
    'recall': recall_score,
    'f1': f1_score,
    'f2_metric': f2_metric,
    'average_precision': average_precision_score
}

# Store results
final_scores = {}

for name, fn in metrics.items():
    if name == 'average_precision':
        score = fn(y_test, y_test_prob)  # Needs probabilities
    else:
        score = fn(y_test, y_test_pred)
    final_scores[name] = score

# Display results
print("Final Model Evaluation on Test Set:")
for name, score in final_scores.items():
    print(f"{name}: {score:.4f}")

print("\nTesting classification report:")
print(classification_report(y_test, y_test_pred, digits=4))

f2_class_0 = fbeta_score(y_test, y_test_pred, beta=2, pos_label=0)
print(f"F2-score for class 0: {f2_class_0:.4f}")

Final Model Evaluation on Test Set:
accuracy: 0.7798
precision: 0.1434
recall: 0.6863
f1: 0.2373
f2_metric: 0.3906
average_precision: 0.1265

Testing classification report:
              precision    recall  f1-score   support

           0     0.9794    0.7848    0.8714       971
           1     0.1434    0.6863    0.2373        51

    accuracy                         0.7798      1022
   macro avg     0.5614    0.7355    0.5543      1022
weighted avg     0.9377    0.7798    0.8397      1022

F2-score for class 0: 0.8172


In [12]:
# Dummy Training Performance
dummy = DummyClassifier(strategy='uniform')
preprocessor = FullPreprocessor()

dummy_perf, dummy_pred = smote_cross_validate_and_predict(dummy, preprocessor, X_train, y_train, scoring_metrics)

metric_names = ['accuracy', 'precision', 'recall', 'f1', 'f2_metric', 'average_precision', 'prec_0', 'rec_0', 'f1_0', 'f2_0']
mean_metrics = dummy_perf[metric_names].mean()

results = []
results.append({
    **mean_metrics.to_dict()
})

results_df = pd.DataFrame(results)
results_df.head()


,accuracy,precision,recall,f1,f2_metric,average_precision,prec_0,rec_0,f1_0,f2_0
0,0.481166,0.049452,0.534211,0.090512,0.180385,0.050519,0.953075,0.478406,0.636731,0.531216


In [13]:
# Dummy testing performance

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, average_precision_score

final_dummy = dummy = DummyClassifier(strategy='uniform')
preprocessor = FullPreprocessor()

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

X_train_bal, y_train_bal = SMOTE(random_state=1234).fit_resample(X_train_proc, y_train)

final_dummy.fit(X_train_bal, y_train_bal)

# Your predicted labels
y_train_pred = final_dummy.predict(X_train_bal)
y_test_pred = final_dummy.predict(X_test_proc)

# For average_precision_score, we need predicted probabilities
y_test_prob = final_dummy.predict_proba(X_test_proc)[:, 1]  # Prob for class 1

# Build a name-function mapping (handles string and callable metrics)
metrics = {
    'accuracy': accuracy_score,
    'precision': precision_score,
    'recall': recall_score,
    'f1': f1_score,
    'f2_metric': f2_metric,
    'average_precision': average_precision_score
}

# Store results
final_scores = {}

for name, fn in metrics.items():
    if name == 'average_precision':
        score = fn(y_test, y_test_prob)  # Needs probabilities
    else:
        score = fn(y_test, y_test_pred)
    final_scores[name] = score

# Display results
print("Final Model Evaluation on Test Set:")
for name, score in final_scores.items():
    print(f"{name}: {score:.4f}")

print("\nTesting classification report:")
print(classification_report(y_test, y_test_pred, digits=4))

f2_class_0 = fbeta_score(y_test, y_test_pred, beta=2, pos_label=0)
print(f"F2-score for class 0: {f2_class_0:.4f}")

## TODO:
# - Evaluate SMOTE vs RandomOverSampler
# - Evaluate SmokingStatusEncoder vs an Ordinal Encoder

Final Model Evaluation on Test Set:
accuracy: 0.5049
precision: 0.0565
recall: 0.5686
f1: 0.1028
f2_metric: 0.2022
average_precision: 0.0499

Testing classification report:
              precision    recall  f1-score   support

           0     0.9568    0.5015    0.6581       971
           1     0.0565    0.5686    0.1028        51

    accuracy                         0.5049      1022
   macro avg     0.5067    0.5351    0.3805      1022
weighted avg     0.9119    0.5049    0.6304      1022

F2-score for class 0: 0.5543
